In [ ]:
# In[1]:

import pandas as pd

# Load files
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds)
for df in (metric_df, trace_df, log_df, error_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) Overviews: earliest, latest, total rows
def make_overview(df):
    return pd.DataFrame([{
        'earliest_utc': df['timestamp'].min(),
        'latest_utc': df['timestamp'].max(),
        'row_count': len(df)
    }])

metric_overview = make_overview(metric_df)
trace_overview = make_overview(trace_df)
log_overview = make_overview(log_df)
error_overview = make_overview(error_df)

# 2) Unique cmdb_id values and their counts (top 20)
metric_cmdb_counts = metric_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(20)
trace_cmdb_counts = trace_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(20)
log_cmdb_counts = log_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(20)
error_cmdb_counts = error_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(20)

# 3) metric.csv: per (cmdb_id, kpi_name) stats: count, mean, P50, P90, P95 -> top 50 by count
metric_grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
metric_kpi_stats = metric_grp.agg(
    count='count',
    mean='mean',
    p50=lambda x: x.quantile(0.5),
    p90=lambda x: x.quantile(0.9),
    p95=lambda x: x.quantile(0.95)
).reset_index().sort_values('count', ascending=False)
metric_kpi_stats = metric_kpi_stats.round(6).head(50)

# 4) trace.csv: per (cmdb_id, trace_name) stats: count, mean, P50, P90, P95 -> top 50 by count
trace_grp = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
trace_stats = trace_grp.agg(
    count='count',
    mean='mean',
    p50=lambda x: x.quantile(0.5),
    p90=lambda x: x.quantile(0.9),
    p95=lambda x: x.quantile(0.95)
).reset_index().sort_values('count', ascending=False)
trace_stats = trace_stats.round(6).head(50)

# 5) log.csv: per (cmdb_id, log_name) stats: count, mean, P95 -> top 50 by count
log_grp = log_df.groupby(['cmdb_id', 'log_name'])['value']
log_stats = log_grp.agg(
    count='count',
    mean='mean',
    p95=lambda x: x.quantile(0.95)
).reset_index().sort_values('count', ascending=False)
log_stats = log_stats.round(6).head(50)

# 6) error_logs.csv: top 20 rows of (timestamp ISO UTC, cmdb_id, message)
error_top20 = error_df[['timestamp', 'cmdb_id', 'message']].copy()
# Format timestamp in UTC ISO (Z)
error_top20['timestamp'] = error_top20['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
error_top20 = error_top20.head(20)

# Keep outputs compact as requested
metric_overview, metric_cmdb_counts, metric_kpi_stats, trace_overview, trace_cmdb_counts, trace_stats, log_overview, log_cmdb_counts, log_stats, error_overview, error_cmdb_counts, error_top20

```
Out[1]:
```
```python
# Build a plain-English summary from previously-loaded results (uses existing variables in the kernel)
metric_rows = int(metric_overview.loc[0, 'row_count'])
metric_earliest = metric_overview.loc[0, 'earliest_utc']
metric_latest = metric_overview.loc[0, 'latest_utc']

trace_rows = int(trace_overview.loc[0, 'row_count']) if 'trace_overview' in globals() and not trace_overview['row_count'].isnull().any() else 0
log_rows = int(log_overview.loc[0, 'row_count'])
error_rows = int(error_overview.loc[0, 'row_count']) if 'error_overview' in globals() and not error_overview['row_count'].isnull().any() else 0

top_metric_cmdb = ", ".join([f"{r['cmdb_id']}({r['count']})" for _, r in metric_cmdb_counts.head(5).iterrows()])
top_log_cmdb = ", ".join([f"{r['cmdb_id']}({r['count']})" for _, r in log_cmdb_counts.head(5).iterrows()])

# Select a few representative metric KPI rows (top by count) for compact reporting
sample_metric_kpis = metric_kpi_stats.head(5).copy()
sample_lines = []
for _, r in sample_metric_kpis.iterrows():
    sample_lines.append(f"{r['cmdb_id']}/{r['kpi_name']}: count={int(r['count'])}, mean={r['mean']:.3g}, p95={r['p95']:.3g}")

# Compose summary
summary = (
    "Summary of telemetry (UTC):\n"
    f"- Metrics: {metric_rows} rows from {metric_earliest} to {metric_latest}.\n"
    f"  Top components by metric row count (top 5): {top_metric_cmdb}.\n"
    f"  Representative top metric KPIs:\n    - " + "\n    - ".join(sample_lines) + "\n"
    f"- Traces: {trace_rows} rows. (No trace data available.)\n"
    f"- Logs: {log_rows} rows. Top components by log row count (top 5): {top_log_cmdb}.\n"
    f"- Error logs: {error_rows} rows. (No error_log records available.)\n\n"
    "Notable observations from the metric KPIs:\n"
    "- Several services show memory and socket-related metrics (e.g., carts and carts-db have high mem/socket mean values).\n"
    "- orders-db and catalogue-db show high diskio magnitudes with large p95 values (suggesting bursty/high disk I/O).\n"
    "- rabbitmq shows elevated CPU p95 compared with its median (indicative of occasional CPU spikes).\n\n"
    "Overall conclusion:\n"
    "- Telemetry is available for metrics and logs in the window shown; trace and error-log files are empty.\n"
    "- Key areas to investigate further (based on metric p95 and means): database disk I/O (orders-db, catalogue-db), memory/socket usage on carts and carts-db, and occasional CPU spikes on rabbitmq.\n"
    "- Because traces and error messages are not available, follow-up actions should include enabling trace/error logging for the incident window and drilling into the listed components' full metric series.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               earliest_utc                latest_utc  row_count
0 2024-01-21 07:20:00+00:00 2024-01-21 07:44:00+00:00       1803,               cmdb_id  count
0              orders    167
1           front-end    157
2               carts    151
3           catalogue    151
4                user    151
5             payment    137
6            shipping    135
7           orders-db     98
8        catalogue-db     98
9            carts-db     98
10       queue-master     98
11            user-db     98
12           rabbitmq     97
13         session-db     93
14  rabbitmq-exporter     74,          cmdb_id kpi_name  count          mean           p50           p90           p95
4          carts      mem     25  2.213194e+08  2.155160e+08  2.552660e+08  2.905056e+08
5          carts   socket     25  1.230274e+01  1.235000e+01  1.453103e+01  1.471035e+01
10      carts-db   socket     25  6.874667e+00  7.000000e+00  7.750000e+00  8.263333e+00
9       carts-db      mem     25  9.365718e+07  9.424350e+07  1.161124e+08  1.379711e+08
45       payment   socket     25  5.638000e+00  6.000000e+00  6.000000e+00  6.933333e+00
..           ...      ...    ...           ...           ...           ...           ...
19  catalogue-db   diskio     24  1.123586e+05  0.000000e+00  0.000000e+00  1.008776e+06
18  catalogue-db      cpu     24  5.241700e-01  1.905820e-01  2.009600e-01  7.181760e-01
38     orders-db   diskio     24  6.843758e+05  8.071156e+05  8.678355e+05  8.970935e+05
37     orders-db      cpu     24  3.839240e-01  3.798760e-01  4.376460e-01  4.462460e-01
51      rabbitmq      cpu     24  1.796752e+00  7.465240e-01  1.087894e+00  5.878960e+00

[50 rows x 7 columns],   earliest_utc latest_utc  row_count
0          NaT        NaT          0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [cmdb_id, trace_name, count, mean, p50, p90, p95]
Index: [],                earliest_utc                latest_utc  row_count
0 2024-01-21 07:20:00+00:00 2024-01-21 07:44:00+00:00        438,               cmdb_id  count
0            shipping     48
1              orders     46
2            carts-db     46
3        queue-master     46
4           front-end     46
5                user     44
6           catalogue     42
7             payment     42
8               carts     38
9          session-db     10
10          orders-db      8
11       catalogue-db      6
12            user-db      6
13           rabbitmq      6
14  rabbitmq-exporter      4,               cmdb_id         log_name  count        mean     p95
25           shipping    log.row_count     24   53.916667   69.55
24           shipping  log.error_count     24    0.000000    0.00
10             orders  log.error_count     23    0.000000    0.00
11             orders    log.row_count     23  106.260870  129.60
3            carts-db    log.row_count     23    5.739130   37.90
..                ...              ...    ...         ...     ...
18           rabbitmq  log.error_count      3    0.000000    0.00
28            user-db  log.error_count      3    0.000000    0.00
29            user-db    log.row_count      3   16.333333   25.30
21  rabbitmq-exporter    log.row_count      2    2.500000    2.95
20  rabbitmq-exporter  log.error_count      2    0.000000    0.00

[30 rows x 5 columns],   earliest_utc latest_utc  row_count
0          NaT        NaT          0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# reuse metric_df if present, else load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# ensure timestamp parsed
if metric_df['timestamp'].dtype == object:
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# 1) Calculate global p95 and p05 per (cmdb_id, kpi_name) using the full series (before any window filtering)
global_thresh = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter metric rows to the incident window (UTC)
start = pd.Timestamp("2024-01-21 07:17:31", tz="UTC")
end = pd.Timestamp("2024-01-21 07:47:31", tz="UTC")
window_df = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# If no rows in window, return empty summary
if window_df.shape[0] == 0:
    anomaly_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','rows_in_window','high_anoms','low_anoms',
        'earliest_anom_utc','max_value_in_window','global_p95','global_p05','total_anoms'
    ])
else:
    # merge global thresholds into windowed rows
    merged = window_df.merge(global_thresh, on=['cmdb_id', 'kpi_name'], how='left')

    # flag anomalies using global thresholds (computed from full series)
    merged['high_anom'] = merged['value'] >= merged['p95']
    merged['low_anom'] = merged['value'] <= merged['p05']
    # prepare anomaly timestamp (timestamp when anomaly else NaT)
    merged['anomaly_ts'] = merged['timestamp'].where(merged['high_anom'] | merged['low_anom'])

    # aggregate per (cmdb_id, kpi_name)
    agg = (
        merged
        .groupby(['cmdb_id', 'kpi_name'])
        .agg(
            rows_in_window = ('value', 'count'),
            high_anoms = ('high_anom', 'sum'),
            low_anoms = ('low_anom', 'sum'),
            earliest_anom_utc = ('anomaly_ts', 'min'),
            max_value_in_window = ('value', 'max'),
            global_p95 = ('p95', 'first'),
            global_p05 = ('p05', 'first')
        )
        .reset_index()
    )

    # compute total anomalies and format earliest timestamp as ISO UTC
    agg['total_anoms'] = agg['high_anoms'] + agg['low_anoms']
    agg['earliest_anom_utc'] = agg['earliest_anom_utc'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

    # sort by total anomalies desc and keep top 20, keep columns requested
    anomaly_summary = (
        agg
        .sort_values(['total_anoms','rows_in_window'], ascending=[False, False])
        .loc[:, ['cmdb_id','kpi_name','rows_in_window','high_anoms','low_anoms',
                 'earliest_anom_utc','max_value_in_window','global_p95','global_p05','total_anoms']]
        .head(20)
    )

    # round numeric columns for compactness
    for c in ['max_value_in_window','global_p95','global_p05']:
        anomaly_summary[c] = anomaly_summary[c].round(6)

anomaly_summary

```
Out[2]:
```
Summary of anomaly analysis (incident window 2024-01-21 07:17:31 – 07:47:31 UTC):

- Highest anomaly counts (top affected (cmdb_id, kpi_name)):
  - carts / diskio — 14 rows in window, 14 high + 14 low anomalies (total 28). earliest anomaly 2024-01-21T07:20:00Z. max value in window = 0.0.
  - catalogue / diskio — 14 rows, 14 high + 14 low (total 28). earliest 2024-01-21T07:20:00Z. max = 0.0.
  - user / error — 14 rows, 14 high + 14 low (total 28). earliest 2024-01-21T07:20:00Z. max = 0.0.

- Many socket-related anomalies across services (large counts):
  - catalogue-db/socket (25 rows): 2 high, 22 low, earliest 07:20, max 6.0, global p95≈5.287, p05=4.0 (total 24).
  - rabbitmq/socket (25 rows): 2 high, 22 low, earliest 07:20, max≈20.667, global p95≈16.927, p05=11.0 (total 24).
  - rabbitmq-exporter/socket, session-db/socket, carts-db/socket, user-db/socket, orders-db/socket, catalogue/socket, payment/socket, shipping/socket — all show nontrivial anomaly counts; several have earliest anomalies around 07:20–07:35 UTC.

- Disk I/O spikes (nonzero maxima and high global p95):
  - catalogue-db/diskio, orders/db diskio, rabbitmq/diskio, session-db/diskio — show large max values and elevated global p95s (indicating bursty/high I/O episodes). earliest anomalies mostly at 07:20.

- Error metrics:
  - front-end/error and orders/error show anomaly counts (earliest anomalies ~07:20, max values >0), indicating increased error-rate signals for those services.

- Important data-quality note: For some metrics (e.g., carts.diskio, catalogue.diskio, user.error) the computed global p95 and p05 are both 0.0. That causes rows with value == 0 to be flagged simultaneously as both "high" and "low" anomalies. Those specific counts should be interpreted with caution — they likely reflect a thresholding artifact rather than true opposite-direction anomalies.

Recommendations / next steps:
- Investigate disk I/O on catalogue-db and orders-db and the components showing very high diskio p95 values.
- Inspect socket-related behavior across catalogue-db, rabbitmq, session-db, carts-db, user-db and orders-db (connections/latency/fd exhaustion).
- Correlate the earliest anomaly times (~07:20 UTC) across components to find common upstream causes (network, broker, or deployment activity).
- Because some metrics have p95==p05==0, validate metric normalization and consider using alternative thresholds or raw-series inspection to avoid double-counting artifacts.
- Enable/collect traces and application error logs for the listed services (if not already available) to pinpoint root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id kpi_name  rows_in_window  high_anoms  low_anoms     earliest_anom_utc  max_value_in_window    global_p95    global_p05  total_anoms
1               carts   diskio              14          14         14  2024-01-21T07:20:00Z         0.000000e+00  0.000000e+00  0.000000e+00           28
12          catalogue   diskio              14          14         14  2024-01-21T07:20:00Z         0.000000e+00  0.000000e+00  0.000000e+00           28
69               user    error              14          14         14  2024-01-21T07:20:00Z         0.000000e+00  0.000000e+00  0.000000e+00           28
21       catalogue-db   socket              25           2         22  2024-01-21T07:20:00Z         6.000000e+00  5.286667e+00  4.000000e+00           24
54           rabbitmq   socket              25           2         22  2024-01-21T07:20:00Z         2.066667e+01  1.692667e+01  1.100000e+01           24
57  rabbitmq-exporter   socket              25           2         22  2024-01-21T07:20:00Z         2.000000e+00  1.580000e+00  1.000000e+00           24
61         session-db   socket              25           2         22  2024-01-21T07:20:00Z         6.000000e+00  4.710000e+00  3.000000e+00           24
19       catalogue-db   diskio              24           2         22  2024-01-21T07:20:00Z         1.509811e+06  1.008776e+06  0.000000e+00           24
52           rabbitmq   diskio              23           2         19  2024-01-21T07:20:00Z         7.159402e+04  3.716433e+04  0.000000e+00           21
30             orders   diskio              19           1         17  2024-01-21T07:20:00Z         3.301170e+05  2.245076e+05  0.000000e+00           18
56  rabbitmq-exporter      mem              25           2         15  2024-01-21T07:20:00Z         9.306112e+06  7.520802e+06  4.538368e+06           17
59         session-db   diskio              19           1         16  2024-01-21T07:20:00Z         1.979915e+04  1.812721e+04  0.000000e+00           17
23          front-end    error              14           1         12  2024-01-21T07:20:00Z         4.257918e+00  1.905481e+00  0.000000e+00           13
31             orders    error              14           1         12  2024-01-21T07:20:00Z         7.390490e+00  4.091486e+00  0.000000e+00           13
10           carts-db   socket              25           2          7  2024-01-21T07:35:00Z         1.000000e+01  8.263333e+00  6.000000e+00            9
78            user-db   socket              25           2          7  2024-01-21T07:35:00Z         1.200000e+01  1.024000e+01  7.000000e+00            9
40          orders-db   socket              25           6          2  2024-01-21T07:33:00Z         8.000000e+00  8.000000e+00  2.086667e+00            8
16          catalogue   socket              25           2          3  2024-01-21T07:35:00Z         8.000000e+00  7.466667e+00  5.000000e+00            5
45            payment   socket              25           2          3  2024-01-21T07:35:00Z         8.000000e+00  6.933333e+00  4.000000e+00            5
66           shipping   socket              25           2          3  2024-01-21T07:23:00Z         6.816667e+00  6.423333e+00  3.000000e+00            5```
```

In [ ]:
# In[3]:

import pandas as pd

# reuse existing log_df if available, else load
if 'log_df' not in globals():
    log_df = pd.read_csv("log.csv")
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
else:
    # ensure timestamp parsed to UTC datetimes
    if log_df['timestamp'].dtype == object or not pd.api.types.is_datetime64tz_dtype(log_df['timestamp']):
        log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# incident window (UTC)
start = pd.Timestamp("2024-01-21 07:17:31", tz="UTC")
end = pd.Timestamp("2024-01-21 07:47:31", tz="UTC")

# filter to window
log_win = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end)].copy()

# If no rows in window, return empty dataframe with requested columns
if log_win.shape[0] == 0:
    log_summary = pd.DataFrame(columns=['cmdb_id','error_count_sum','row_count_sum','log_rows_in_window','earliest_error_utc'])
else:
    # sum of log.error_count per cmdb_id (missing treated as zero)
    err_sum = (
        log_win[log_win['log_name'] == 'log.error_count']
        .groupby('cmdb_id')['value'].sum()
        .rename('error_count_sum')
    )

    # sum of log.row_count per cmdb_id
    row_sum = (
        log_win[log_win['log_name'] == 'log.row_count']
        .groupby('cmdb_id')['value'].sum()
        .rename('row_count_sum')
    )

    # count of log rows found for that cmdb_id in the window
    rows_count = log_win.groupby('cmdb_id').size().rename('log_rows_in_window')

    # earliest UTC timestamp where log.error_count > 0 for that cmdb_id (if any)
    err_pos = log_win[(log_win['log_name'] == 'log.error_count') & (log_win['value'] > 0)]
    earliest_err = err_pos.groupby('cmdb_id')['timestamp'].min().rename('earliest_error_utc')

    # assemble all cmdb_ids present in the window
    cmdb_ids = pd.Index(sorted(log_win['cmdb_id'].unique()), name='cmdb_id')
    summary = pd.DataFrame(index=cmdb_ids).reset_index()

    # merge aggregates, filling missing sums with 0
    summary = summary.merge(err_sum.rename('error_count_sum'), on='cmdb_id', how='left')
    summary = summary.merge(row_sum.rename('row_count_sum'), on='cmdb_id', how='left')
    summary = summary.merge(rows_count.rename('log_rows_in_window'), on='cmdb_id', how='left')
    summary = summary.merge(earliest_err.rename('earliest_error_utc'), on='cmdb_id', how='left')

    summary['error_count_sum'] = summary['error_count_sum'].fillna(0)
    summary['row_count_sum'] = summary['row_count_sum'].fillna(0)
    summary['log_rows_in_window'] = summary['log_rows_in_window'].fillna(0).astype(int)

    # format earliest_error_utc as UTC ISO string, keep empty if none
    summary['earliest_error_utc'] = summary['earliest_error_utc'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

    # select and sort top 20 by descending error_count_sum
    log_summary = summary.loc[:, ['cmdb_id','error_count_sum','row_count_sum','log_rows_in_window','earliest_error_utc']]\
                         .sort_values('error_count_sum', ascending=False)\
                         .head(20).reset_index(drop=True)

    # round numeric sums for compactness
    log_summary['error_count_sum'] = log_summary['error_count_sum'].round(6)
    log_summary['row_count_sum'] = log_summary['row_count_sum'].round(6)

log_summary

```
Out[3]:
```
Summary (incident window 2024-01-21 07:17:31 — 07:47:31 UTC):

- No errors reported in normalized logs: error_count_sum = 0 for every cmdb_id and earliest_error_utc is empty for all services (no log.error_count > 0 found in the window).
- Log volume by service (row_count_sum, selected top entries):
  - front-end: 38153
  - user: 11105
  - queue-master: 10572
  - catalogue: 3058
  - orders: 2444
  - payment: 1828
  - carts: 1322
  - shipping: 1294
  - (others have much lower totals; rabbitmq-exporter has only 5 rows)

Interpretation and recommended next steps:
- The normalized log stream shows no recorded error counts during the incident window. This could mean there truly were no logged errors, or that error logging/normalization/filtering missed them.
- Verify raw application error logs (or error_logs.csv) and the log collection pipeline to ensure errors would have been captured and normalized into log.error_count.
- Since metrics showed anomalies earlier, correlate those metric anomalies with raw logs/traces (if available) because normalized log.error_count did not surface errors.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  error_count_sum  row_count_sum  log_rows_in_window earliest_error_utc
0               carts                0           1322                  38                NaN
1            carts-db                0            132                  46                NaN
2           catalogue                0           3058                  42                NaN
3        catalogue-db                0            215                   6                NaN
4           front-end                0          38153                  46                NaN
5              orders                0           2444                  46                NaN
6           orders-db                0             47                   8                NaN
7             payment                0           1828                  42                NaN
8        queue-master                0          10572                  46                NaN
9            rabbitmq                0            123                   6                NaN
10  rabbitmq-exporter                0              5                   4                NaN
11         session-db                0             27                  10                NaN
12           shipping                0           1294                  48                NaN
13               user                0          11105                  44                NaN
14            user-db                0             49                   6                NaN```
```